In [3]:
# for root anchor (notebook moved into subfolder)
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np

from validation_metrics import (
    ValidationGate, 
    VoltammogramFidelityIndex,
    assign_nearest_log_class,
    quick_compare
)

# centralized paths for all project files
import paths

In [2]:
# run validation gate module on physics informed augmented signals

gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.AUGMENTED_SIGNALS_CSV, run_tiers=[1, 2])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)

print(results['pff_df'].groupby('class_uM')['Wasserstein'].agg(['mean', 'median', 'count', 'max']))
print()
print(results['pff_df'].sort_values('Wasserstein', ascending=False).head(15))

print(results.get('delta_acf_peak', 'not in results dict'))
print([k for k in results.keys() if 'acf' in k.lower()])
print (results['pff_df']['Wasserstein'].describe())


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 0.8669  [Good     ]  ║
║  NEW  P

In [3]:
# --- 1. Load Data ---
target_col = 'concentration'

E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

# Real signals and labels
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
y_real = df_real[target_col].values
X_real = df_real.drop(columns=[target_col]).values

# --- 2. Initialize Gate & Extract Features ---
gate = ValidationGate(E, X_real, y_real)

# ACCESS feat_real
feat_real = gate.feat_real  

# Evaluate synthetic data to get feat_synth
results = gate.evaluate_csv(paths.AUGMENTED_SIGNALS_CSV, target_col=target_col)
feat_synth = results['feat_synth']

df_aug = pd.read_csv(paths.AUGMENTED_SIGNALS_CSV)
y_synth = df_aug[target_col].values


# --- 3. Diagnostics ---

# How many synthetic samples per real class, after binning?
real_classes = np.unique(gate.y_real)
y_synth_binned = assign_nearest_log_class(y_synth, real_classes)
print("--- Synthetic Samples per Class ---")
print(pd.Series(y_synth_binned).value_counts().sort_index())

# Manually reproduce ONE cell -- Ep at 2.50 µM -- to see the raw numbers
r = feat_real[gate.y_real == 2.5]['Ep'].values
s = feat_synth[y_synth_binned == 2.5]['Ep'].values

print("\n--- Ep at 2.50 µM ---")
print('real Ep:', r)
print('synth Ep:', s)
print('real std:', r.std(ddof=1), 'synth std:', s.std(ddof=1))
print('pooled std used in normaliser:', np.std(np.concatenate([r, s]), ddof=1))


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ✅ PASS              ║
║    TSTR l

In [4]:
# timegan test
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.TIMEGAN_SIGNALS_CSV, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.312  (≥0.90)                ║
║    JSD mean / max                     : 0.1300 / 0.5820           ║
║    MMD²                               : 0.192612                ║
║    SWD mean / max                     : 0.2928 / 1.4817           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9523  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.51084  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [5]:
# wgangp test
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results = gate.evaluate_csv(paths.WGANGP_SIGNALS_CSV, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.688  (≥0.90)                ║
║    JSD mean / max                     : 0.0593 / 0.1908           ║
║    MMD²                               : 0.063509                ║
║    SWD mean / max                     : 0.1379 / 0.4227           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.667  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9538  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.01733  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [6]:

gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

results_sanity = gate.evaluate_csv(paths.REAL_SIGNALS_CSV, run_tiers=[1, 2])

vfi_perfect = VoltammogramFidelityIndex.from_gate_results(results_sanity, verbose=True)
print(f"VFI Sanity Check: {vfi_perfect:.4f}") 


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 1.000  (≥0.90)                ║
║    JSD mean / max                     : 0.0000 / 0.0000           ║
║    MMD²                               : -0.024235                ║
║    SWD mean / max                     : 0.0000 / 0.0000           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 1.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9775  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.00000  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 1.0000  [Excellent]  ║
║  NEW  

In [7]:
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': paths.AUGMENTED_SIGNALS_CSV,
    'TimeGAN': paths.TIMEGAN_SIGNALS_CSV,
    'WGANGP': paths.WGANGP_SIGNALS_CSV
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass  final_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True        True
1             TimeGAN        0.3118    0.1300  0.192612    0.2928           0.000       0.9523    0.51084  0.5166      Poor            0.6625       False       False       False
2              WGANGP        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849      Good            0.7610       False       False       False


In [8]:
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': paths.AUGMENTED_SIGNALS_CSV,
    'TimeGAN': paths.TIMEGAN_SIGNALS_CSV,
    'WGANGP': paths.WGANGP_SIGNALS_CSV
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2, 3, 4])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass  TSTR_log_rmse_ratio  tier3_pass  mean_DS  tier4_pass  final_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True               0.7746        True   0.8884        True        True
1             TimeGAN        0.3118    0.1300  0.192612    0.2928           0.000       0.9523    0.51084  0.5166      Poor            0.6625       False       False               1.7695       False   0.7037        True       False
2              WGANGP        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849      Good            0.7610       False       False               0.8133       False   0.8880        True       False


# NEW: all tier-urilor de date
Above only physics-aug, TimeGAN-real, WGAN-GP-real tested + sanity check only on 1-2. Fleshed out below for documentation


1. sanity check for all 4 tiers
2. test `combined` datasets (real + physics-aug), used in `run_condition_sweep.py`;
3. test GAN thatw ere trained on combined dataset (`timegan_combined`, `wgangp_combined`)
4. test comparativ general

In [ ]:
# NEW: sanity check complet, toate cele 4 niveluri
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)
results_sanity_full = gate.evaluate_csv(paths.REAL_SIGNALS_CSV, run_tiers=[1, 2, 3, 4])
vfi_perfect_full = VoltammogramFidelityIndex.from_gate_results(results_sanity_full, verbose=True)

print(f"VFI Sanity Check (toate 4 niveluri): {vfi_perfect_full:.4f}")
print(f"final_pass: {results_sanity_full['final_pass']}")
print(f"mean_DS (asteptat ~0.5, discriminatorul nu poate distinge real de el insusi): {results_sanity_full.get('mean_ds')}")


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 1.000  (≥0.90)                ║
║    JSD mean / max                     : 0.0000 / 0.0000           ║
║    MMD²                               : -0.024235                ║
║    SWD mean / max                     : 0.0000 / 0.0000           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 1.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9775  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.00000  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ✅ PASS              ║
║    TSTR 

In [ ]:
# NEW: testare tier 'combined' (real + physics-aug)
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)
results_combined = gate.evaluate_csv(paths.COMBINED_SIGNALS_CSV, run_tiers=[1, 2, 3, 4])
vfi_combined = VoltammogramFidelityIndex.from_gate_results(results_combined, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.997  (≥0.90)                ║
║    JSD mean / max                     : 0.0590 / 0.2002           ║
║    MMD²                               : 0.017775                ║
║    SWD mean / max                     : 0.1387 / 0.2837           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9793  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.08359  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ✅ PASS              ║
║    TSTR l

In [ ]:
# NEW: gan trained on combined
gate = ValidationGate.from_csv(paths.POTENTIAL_GRID_CSV, paths.REAL_SIGNALS_CSV)

print("=== TimeGAN (antrenat pe date combinate) ===")
results_timegan_comb = gate.evaluate_csv(paths.TIMEGAN_SIGNALS_TRAINED_COMBINED_CSV, run_tiers=[1, 2, 3, 4])
VoltammogramFidelityIndex.from_gate_results(results_timegan_comb, verbose=True)

print("\n=== WGAN-GP (antrenat pe date combinate) ===")
results_wgangp_comb = gate.evaluate_csv(paths.WGANGP_SIGNALS_TRAINED_COMBINED_CSV, run_tiers=[1, 2, 3, 4])
VoltammogramFidelityIndex.from_gate_results(results_wgangp_comb, verbose=True)

=== TimeGAN (antrenat pe date combinate) ===



╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.450  (≥0.90)                ║
║    JSD mean / max                     : 0.1088 / 0.5290           ║
║    MMD²                               : 0.132346                ║
║    SWD mean / max                     : 0.1489 / 0.4816           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.167  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9847  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.20179  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.574  (≥0.90)                ║
║    JSD mean / max                     : 0.0719 / 0.2111           ║
║    MMD²                               : 0.039600                ║
║    SWD mean / max                     : 0.1439 / 0.4559           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.333  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9729  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.06285  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

0.7772162794634514

In [ ]:
# NEW: general
df_real = pd.read_csv(paths.REAL_SIGNALS_CSV)
E = pd.read_csv(paths.POTENTIAL_GRID_CSV).values.flatten()
y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

all_files_to_test = {
    'physics_aug':                        paths.AUGMENTED_SIGNALS_CSV,
    'combined_real_physaug':               paths.COMBINED_SIGNALS_CSV,
    'timegan_real':                        paths.TIMEGAN_SIGNALS_CSV,
    'wgangp_real':                         paths.WGANGP_SIGNALS_CSV,
    'timegan_combined':                    paths.TIMEGAN_SIGNALS_TRAINED_COMBINED_CSV,
    'wgangp_combined':                     paths.WGANGP_SIGNALS_TRAINED_COMBINED_CSV,
    'gan_only_600 (wgangp+timegan real)':  paths.GAN_ONLY_RAW_SIGNALS_CSV,
    'combined_all_940 (real+physaug+gan)': paths.COMBINED_ALL_RAW_SIGNALS_CSV,
}

all_batches = {}
for name, path in all_files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    all_batches[name] = (X_synth, y_synth)

master_comparison_df = quick_compare(E, X_real, y_real, all_batches, run_tiers=[1, 2, 3, 4])
master_comparison_df = master_comparison_df.sort_values('VFI', ascending=False).reset_index(drop=True)

master_comparison_df.to_csv(paths.RESULTS_DIR / 'validation_gate_all_tiers_comparison.csv', index=False)
print(master_comparison_df.to_string())

                                 batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI  VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass  TSTR_log_rmse_ratio  tier3_pass  mean_DS  tier4_pass  final_pass
0                combined_real_physaug        0.9973    0.0590  0.017775    0.1387           0.833       0.9793    0.08359  0.9016  Excellent            0.7461        True        True               0.6689        True   0.9429        True        True
1                          wgangp_real        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849       Good            0.7610       False       False               0.8133       False   0.8880        True       False
2                          physics_aug        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669       Good            0.7072        True        True               0.7746        True   0.8884        True        True


In [ ]:
import plotly.express as px
import plot_style

label_colors = {
    'Excellent': '#2ca02c', 
    'Good': '#98df8a', 
    'Marginal': '#ff7f0e', 
    'Poor': '#d62728'
}

fig = px.bar(
    master_comparison_df,
    x='VFI',
    y='batch',
    color='VFI_label',
    color_discrete_map=label_colors,
    orientation='h'
)

# coresponds to ax.invert_yaxis() from Matplotlib
fig.update_yaxes(autorange="reversed")

# praguri
# add_vline ~ axvline
fig.add_vline(x=0.90, line_dash="dot", line_color="black", 
              annotation_text="Excellent", annotation_position="bottom right")
fig.add_vline(x=0.75, line_dash="dot", line_color="gray", 
              annotation_text="Good", annotation_position="bottom right")
fig.add_vline(x=0.60, line_dash="dash", line_color="gray", 
              annotation_text="Marginal", annotation_position="bottom right")

# layout
fig = plot_style.apply_default_plotly_layout(
    fig,
    title_text='VFI across all data tiers (validation gate, tiers 1-4)',
    xaxis_title='Voltammogram Fidelity Index (VFI)',
    yaxis_title='Batch'
)


fig.show()